# 02 — Regionalización de parámetros

Reparte parámetros del **SAND Nacional** entre las 7 regiones usando el archivo de
participaciones. Formatos admitidos (se detectan automáticamente):

1. **Canónico**: `Parámetro | TECHNOLOGY | FUEL | Región | 2022..2055` (o columna
   `Participacion` constante en el tiempo).
2. **Regiones anchas por año**: `Fuel | Anio | Antioquia..Suroccidente` (o
   `Technology | Anio | ...`). Sin columna Parámetro: se infiere que aplica a
   `PARAMETROS_A_REGIONALIZAR`; si un parámetro no se indexa por esa dimensión
   (según `config_depurado.yaml`) se **advierte** y sus filas se descartan.
3. **Regiones anchas constantes**: `Parametro | Fuel/Tecnologia | AN SE IN NE CA OR SO`.
4. **Solo regiones** (`AN..SO`, una fila): participación **comodín** `*` que aplica a
   todos los combos de `PARAMETROS_A_REGIONALIZAR`, manteniendo los códigos del nacional.

Reglas de reparto:

- **Aditivos**: `valor_regional = valor_nacional × participación`
  (los centinelas 99999/solo-9s se copian sin repartir).
- **Intensivos**: `valor_regional = valor_nacional` en cada región donde el código exista.
- Antes de aplicar se valida que las participaciones sumen ≈ 1.0 por combo y año.

Salidas: un **SAND reducido** por parámetro en `SANDs_Reducidos/` (o consolidado)
y un **log** de ejecución (Excel + texto) en `reportes/`.

> Ejecutable de principio a fin con *Run All*.
>
> *Nota:* `config/participaciones_industriales.xlsx` se genera desde el archivo
> legado de participaciones con `python scripts/generar_participaciones_industriales.py`.

In [ ]:
# --- Setup: raíz del proyecto, módulos src/ y configuración ---
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

RAIZ = Path.cwd().resolve()
if not (RAIZ / "config").exists():   # ejecutado desde notebooks/
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import comparador
import regionalizador
import reporte
import sand_io
import utils
import yaml_parser

utils.configurar_logging()

params_cfg = yaml_parser.cargar_params_config(RAIZ / "config" / "params_config.yaml")
paths = yaml_parser.cargar_paths_config(RAIZ / "config" / "paths_config.yaml", raiz=RAIZ)
otoole = yaml_parser.cargar_config_otoole(paths["escenario_nacional"]["config_yaml"])

print(f"Nacional : {paths['escenario_nacional']['sand'].name}")
print(f"Regional : {paths['escenario_regional']['sand'].name}")
print(f"Config otoole: {len(otoole['param'])} parámetros, {len(otoole['set'])} sets")
print(f"Intensivos: {len(params_cfg['parametros_intensivos'])} | Centinela: {params_cfg['valor_centinela']}")

## 1. Configuración de la corrida

In [ ]:
PARAMETROS_A_REGIONALIZAR = [
    "AccumulatedAnnualDemand",
    "TotalTechnologyAnnualActivityLowerLimit",
]
# Filtros de códigos nacionales. None => usar exactamente los combos definidos
# en el archivo de participaciones (recomendado); o listas manuales, p. ej.:
#   TECNOLOGIAS_FILTRO = ["DEMINDNGSBOI_LOW", "DEMINDNGSBOI_MID"]
# Con participación comodín ('*') no hay combos definidos: se regionaliza todo
# el parámetro salvo que se den filtros manuales.
TECNOLOGIAS_FILTRO = None
FUELS_FILTRO = None
MODO_FILTRO = "exacto"                  # 'exacto' | 'contiene'
YEARS_FILTRO = list(range(2022, 2056))  # o subconjunto, ej. range(2022, 2031)

CONSOLIDADO = False                     # True => un solo SAND_Consolidado_*.xlsx
DESCRIPCION = "Industrial"              # sufijo de los archivos de salida
# Admite el formato canónico o los formatos anchos (ver descripción arriba), p. ej.:
#   RUTA_PARTICIPACIONES = RAIZ / "Insumos" / "Participacion_Fuel_RES.xlsx"
RUTA_PARTICIPACIONES = paths["participaciones"]["archivo"]

## 2. Carga de insumos y validación de participaciones

In [ ]:
df_nacional = sand_io.cargar_sand(paths["escenario_nacional"]["sand"])
df_regional = sand_io.cargar_sand(paths["escenario_regional"]["sand"])
# parametros + params_otoole permiten inferir el parámetro en archivos sin esa
# columna y validar la indexación (advertencia + descarte si no corresponde)
df_pct = regionalizador.cargar_participaciones(
    RUTA_PARTICIPACIONES,
    parametros=PARAMETROS_A_REGIONALIZAR,
    params_otoole=otoole["param"],
)

print(f"Nacional: {len(df_nacional):,} filas | Regional: {len(df_regional):,} filas")
display(df_pct.head(8))

malas = regionalizador.validar_participaciones(df_pct, params_cfg["tolerancia_participacion"])
if malas.empty:
    print("OK: todas las participaciones suman ~1.0 entre regiones")
else:
    display(Markdown(f"**ALERTA: {len(malas)} grupos no suman 1.0**"))
    display(malas.head(20))

## 3. Regionalización

Por cada fila nacional que pase el filtro: se verifica que el código exista en el
regional con algún prefijo (si no, se **omite** y queda en el log); si el combo no
está en el archivo de participaciones se lanza un **error descriptivo**.

In [ ]:
sands, logs = {}, []
for parametro in PARAMETROS_A_REGIONALIZAR:
    pct_p = df_pct[df_pct["Parametro"] == parametro]
    es_intensivo = parametro in params_cfg["parametros_intensivos"]
    if pct_p.empty and not es_intensivo:
        print(f"ADVERTENCIA: {parametro} sin filas de participación aplicables; se omite")
        continue
    # El comodín '*' no es un código: no entra a los filtros derivados
    # (filtros vacíos => se regionaliza todo el parámetro con la participación global)
    techs = TECNOLOGIAS_FILTRO if TECNOLOGIAS_FILTRO is not None else sorted(
        c for c in pct_p["TECHNOLOGY"].dropna().unique() if c != regionalizador.COMODIN)
    fuels = FUELS_FILTRO if FUELS_FILTRO is not None else sorted(
        c for c in pct_p["FUEL"].dropna().unique() if c != regionalizador.COMODIN)
    res = regionalizador.regionalizar(
        df_nacional, df_regional, df_pct,
        parametros=[parametro],
        params_otoole=otoole["param"], cfg=params_cfg,
        tecnologias_filtro=techs, fuels_filtro=fuels,
        modo_filtro=MODO_FILTRO, years_filtro=YEARS_FILTRO,
    )
    sands.update(res["sands"])
    logs.append(res["log"])

log = pd.concat(logs, ignore_index=True)
display(Markdown("**Log de regionalización**"))
display(log)

In [ ]:
for parametro, df_sand in sands.items():
    display(Markdown(f"**{parametro}** — {len(df_sand)} filas SAND"))
    display(df_sand.head(8))

## 4. Gráficas de control

In [ ]:
if "AccumulatedAnnualDemand" in sands:
    reporte.grafica_participacion_regional(
        sands["AccumulatedAnnualDemand"], anio=2030, col_dimension="FUEL",
        titulo="AccumulatedAnnualDemand regional por FUEL — 2030",
    )

In [ ]:
if "TotalTechnologyAnnualActivityLowerLimit" in sands:
    reporte.grafica_participacion_regional(
        sands["TotalTechnologyAnnualActivityLowerLimit"], anio=2030, col_dimension="TECHNOLOGY",
        titulo="LowerLimit regional por tecnología — 2030",
    )

## 5. Exportar SANDs reducidos y log

In [ ]:
rutas_sand = regionalizador.escribir_sands(
    sands, paths["outputs"]["sands_reducidos"], DESCRIPCION, consolidado=CONSOLIDADO,
)
for ruta in rutas_sand:
    print(f"SAND : {ruta}")

ruta_log_xlsx, ruta_log_txt = reporte.escribir_log_regionalizacion(
    log, paths["outputs"]["reportes"],
)
print(f"Log  : {ruta_log_xlsx}")
print(f"Log  : {ruta_log_txt}")

## Siguiente paso

Integrar los SAND reducidos al escenario regional y convertirlo con otoole:

```bash
otoole convert csv excel CSV_Regional salida.xlsx config_depurado.yaml
```

*Smoke tests de los módulos: `python tests/test_smoke.py` desde la raíz del proyecto.*